In [1]:
import torch
import pandas as pd
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    pipeline
)

# Load fine-tuned model
fine_tuned_path = "amharic-ner-model"
device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Device:", device)


/Users/mikiyasegaye/MK_Lab/10 Academy/Amharic-E-commerce-Data-Extractor/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps


In [2]:
def compare_models_on_texts(texts, label_list):
    base_model_name = "xlm-roberta-base"

    # Base model
    base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
    base_model = AutoModelForTokenClassification.from_pretrained(
        base_model_name, num_labels=len(label_list)
    )
    ner_base = pipeline("ner", model=base_model, tokenizer=base_tokenizer, aggregation_strategy="simple")

    # Fine-tuned model
    fine_tokenizer = AutoTokenizer.from_pretrained(fine_tuned_path)
    fine_model = AutoModelForTokenClassification.from_pretrained(fine_tuned_path)
    ner_fine = pipeline("ner", model=fine_model, tokenizer=fine_tokenizer, aggregation_strategy="simple")

    # Run and print
    for text in texts:
        print("=" * 60)
        print(f"Input Text: {text}\n")

        print("🔹 Base Model Prediction:")
        base_output = ner_base(text)
        print(base_output if base_output else "No entities found.")

        print("\n🔸 Fine-Tuned Model Prediction:")
        fine_output = ner_fine(text)
        print(fine_output if fine_output else "No entities found.")
        print("\n")


In [6]:
texts = [
    "አዲስ ልብስ በቦሌ ላይ በ 250 ብር ሽያጭ ይካሄዳል።",
    "የቤት እቃ ዋጋ 1500 ብር በሀዋሳ።",
]

# Use label_list from training notebook if available
label_list = ['B-LOC', 'B-ORG', 'B-PER', 'B-TIME', 'B-TTL', 'I-LOC', 'I-ORG', 'I-PER', 'I-TIME', 'I-TTL', 'O']
compare_models_on_texts(texts, label_list)


Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use mps:0
Device set to use mps:0


Input Text: አዲስ ልብስ በቦሌ ላይ በ 250 ብር ሽያጭ ይካሄዳል።

🔹 Base Model Prediction:
[{'entity_group': 'LABEL_5', 'score': np.float32(0.13676082), 'word': 'አዲስ ልብስ በቦሌ ላይ በ 250 ብር ሽያጭ ይካሄዳል።', 'start': 0, 'end': 34}]

🔸 Fine-Tuned Model Prediction:
No entities found.


Input Text: የቤት እቃ ዋጋ 1500 ብር በሀዋሳ።

🔹 Base Model Prediction:
[{'entity_group': 'LABEL_5', 'score': np.float32(0.13830715), 'word': 'የቤት እቃ ዋጋ 1500 ብር በሀዋሳ።', 'start': 0, 'end': 23}]

🔸 Fine-Tuned Model Prediction:
No entities found.


